# Day 1: Random Forest Baseline
## Healthcare Fraud Detection

**Goal**: Establish baseline performance
**Expected**: F1≈0.86, AUROC≈0.97

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/healthcare-fraud-detection')
print(f'✅ Working in: {os.getcwd()}')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import *
import joblib
print('✅ Libraries loaded')

In [ ]:
print('Loading data...')
df = pd.read_csv('data/raw/healthcare_fraud_train.csv')
df['ClaimStartDt'] = pd.to_datetime(df['ClaimStartDt'])
df['ClaimEndDt'] = pd.to_datetime(df['ClaimEndDt'])
print(f'✅ Loaded {len(df):,} claims')
print(f'Fraud rate: {df["PotentialFraud"].mean()*100:.1f}%')

In [ ]:
print('Feature engineering...')
df['ClaimDuration'] = (df['ClaimEndDt'] - df['ClaimStartDt']).dt.days
df['ClaimMonth'] = df['ClaimStartDt'].dt.month
df['ReimbursementRatio'] = df['InscClaimAmtReimbursed'] / (df['ClaimAmount'] + 1)
df['DeductibleRatio'] = df['DeductibleAmtPaid'] / (df['ClaimAmount'] + 1)
df['ClaimPerDay'] = df['ClaimAmount'] / (df['ClaimDuration'] + 1)

provider_stats = df.groupby('ProviderID').agg({'ClaimAmount': ['mean', 'std', 'count'], 'PotentialFraud': 'sum'}).reset_index()
provider_stats.columns = ['ProviderID', 'Provider_AvgClaim', 'Provider_StdClaim', 'Provider_TotalClaims', 'Provider_FraudCount']
provider_stats['Provider_FraudRate'] = provider_stats['Provider_FraudCount'] / provider_stats['Provider_TotalClaims']
df = df.merge(provider_stats, on='ProviderID', how='left')

bene_stats = df.groupby('BeneID').agg({'ClaimAmount': ['mean', 'count']}).reset_index()
bene_stats.columns = ['BeneID', 'Bene_AvgClaim', 'Bene_ClaimCount']
df = df.merge(bene_stats, on='BeneID', how='left')

print('✅ Features created')

In [ ]:
features = ['ClaimAmount', 'InscClaimAmtReimbursed', 'DeductibleAmtPaid', 'ClaimDuration', 
            'ClaimMonth', 'ReimbursementRatio', 'DeductibleRatio', 'ClaimPerDay',
            'Provider_AvgClaim', 'Provider_StdClaim', 'Provider_FraudRate',
            'Bene_AvgClaim', 'Bene_ClaimCount', 'ChronicCond_Diabetes', 'ChronicCond_Heartfailure', 'Gender']

X = df[features].fillna(0)
y = df['PotentialFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
print('Training Random Forest...')
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]

rf_f1 = f1_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_pred_proba)

print('\n' + '='*70)
print('BASELINE RESULTS')
print('='*70)
print(f'F1-Score: {rf_f1:.4f} ⭐')
print(f'ROC-AUC:  {rf_auc:.4f} ⭐')
print('\n' + classification_report(y_test, rf_pred, target_names=['Normal', 'Fraud']))
print(confusion_matrix(y_test, rf_pred))

In [ ]:
import os
os.makedirs('src/models', exist_ok=True)
os.makedirs('results', exist_ok=True)
joblib.dump(rf_model, 'src/models/random_forest_baseline.pkl')
results_day1 = {'model': 'Random Forest', 'f1': rf_f1, 'auc': rf_auc}
joblib.dump(results_day1, 'results/day1_results.pkl')
print('✅ Day 1 Complete!')